# Sentinel-2 Multi-Footprint SIF Model

Train a spatial CNN/U-Net using 6 km, 20 m Sentinel-2 predictor chips with multiple OCO-2 footprint-level supervision targets per chip.

The model predicts a 20 m SIF map. Each predicted map is averaged independently over every fractional OCO-2 footprint mask, and those footprint predictions are compared with observed `target_modis_sif`. Footprint losses are averaged within each chip before averaging across the batch.

## 1. Imports

In [ ]:
from pathlib import Path
from collections import OrderedDict
import hashlib
import json
import math
import random

import altair as alt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Sampler

alt.data_transformers.disable_max_rows()
pd.set_option('display.max_columns', 100)

## 2. Configuration

In [ ]:
# Edit CHIP_DIR after attaching the Kaggle dataset.
CHIP_DIR = Path(
    '/kaggle/input/CHANGE_ME/multisif_6km_20m_indices_fapar_active_crop'
)
METADATA_PATH = CHIP_DIR / 'chip_metadata.csv'
FOOTPRINT_METADATA_PATH = CHIP_DIR / 'footprint_metadata.csv'

OUTPUT_DIR = Path('/kaggle/working/sentinel2_multisif_model')
STATS_CACHE_DIR = Path('/kaggle/working/normalization_stats')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STATS_CACHE_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = 'target_modis_sif'
FINAL_CHECK_COL = 'final_check_modis_sif'

SEED = 42
TRAIN_FRAC = 0.80
VAL_FRAC = 0.10
TEST_FRAC = 0.10

BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4
BASE_CHANNELS = 16
EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
HUBER_BETA = 1.0

EARLY_STOPPING_PATIENCE = 7
LR_PATIENCE = 3
LR_FACTOR = 0.5
MIN_LEARNING_RATE = 1e-6

NUM_WORKERS = 2
SHARD_CACHE_SIZE = 8
MAX_SAMPLES_FOR_STATS = 256
USE_STATS_CACHE = True
NORMALIZE_TARGET = True
USE_AMP = True

MIN_VALID_FOOTPRINTS = 4
SIF_BIN_EDGES = [-0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 1.0, 1.25]
SIF_BIN_LABELS = [
    '[-0.5,-0.25)', '[-0.25,0)', '[0,0.25)', '[0.25,0.5)',
    '[0.5,0.75)', '[0.75,1)', '[1,1.25]'
]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = bool(USE_AMP and DEVICE.type == 'cuda')

print('device:', DEVICE)
print('AMP enabled:', AMP_ENABLED)
print('effective batch size:', BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)

## 3. Index Shards and Load Metadata

In [ ]:
def decode_strings(values: np.ndarray) -> list[str]:
    output = []
    for value in values:
        if isinstance(value, bytes):
            output.append(value.decode('utf-8'))
        else:
            output.append(str(value))
    return output


def build_shard_index(chip_dir: Path) -> tuple[pd.DataFrame, list[str], list[str], list[str]]:
    shard_files = sorted(chip_dir.glob('chips_*.npz'))
    if not shard_files:
        raise FileNotFoundError(f'No chips_*.npz files found in {chip_dir}')

    rows = []
    channel_names = None
    target_names = None
    final_check_names = None
    sample_order = 0

    for shard_id, shard_path in enumerate(shard_files):
        with np.load(shard_path, allow_pickle=False) as shard:
            chip_ids = decode_strings(shard['chip_id'])
            current_channels = decode_strings(shard['channel_names'])
            current_targets = decode_strings(shard['target_names'])
            current_checks = decode_strings(shard['final_check_names'])

        if channel_names is None:
            channel_names = current_channels
            target_names = current_targets
            final_check_names = current_checks
        elif (
            current_channels != channel_names
            or current_targets != target_names
            or current_checks != final_check_names
        ):
            raise ValueError(f'Inconsistent shard schema in {shard_path}')

        for local_index, chip_id in enumerate(chip_ids):
            rows.append({
                'sample_order': sample_order,
                'shard_id': shard_id,
                'shard_path': str(shard_path),
                'local_index': local_index,
                'chip_id': chip_id,
            })
            sample_order += 1

    shard_index = pd.DataFrame(rows)
    if shard_index['chip_id'].duplicated().any():
        raise ValueError(
            'Duplicate chip IDs found across shards. Remove stale NPZ files and rebuild the dataset.'
        )

    return shard_index, channel_names, target_names, final_check_names


metadata = pd.read_csv(METADATA_PATH, parse_dates=['Delta_Date', 'par_date'])
footprint_metadata = pd.read_csv(FOOTPRINT_METADATA_PATH)
shard_index, channel_names, target_names, final_check_names = build_shard_index(CHIP_DIR)

samples = metadata.merge(shard_index, on='chip_id', how='inner', validate='one_to_one')

if TARGET_COL not in target_names:
    raise ValueError(f'{TARGET_COL} is not present in shard targets: {target_names}')

TARGET_INDEX = target_names.index(TARGET_COL)
if len(target_names) != 1 or TARGET_INDEX != 0:
    raise ValueError(f'Expected one stored target ({TARGET_COL}), found {target_names}')
if final_check_names[TARGET_INDEX] != FINAL_CHECK_COL:
    raise ValueError(
        f'Expected {FINAL_CHECK_COL}, found {final_check_names[TARGET_INDEX]}'
    )

print('metadata rows:', len(metadata))
print('indexed shard rows:', len(shard_index))
print('merged samples:', len(samples))
print('channels:', len(channel_names), channel_names)
print('targets:', target_names)
samples.head()

## 4. Filter and Inspect Chips

In [ ]:
samples = samples[
    (samples['chip_status'] == 'inside')
    & (samples['n_valid_footprints'] >= MIN_VALID_FOOTPRINTS)
    & (samples['target_accepted_footprints'] >= 1)
].copy()

samples['month'] = samples['Delta_Date'].dt.month.astype(int)
samples['measurement_mode'] = pd.to_numeric(samples['measurement_mode'], errors='coerce')
samples = samples.sort_values(
    ['sif_year', 'sif_doy', 'mgrs_tile_t', 'chip_id']
).reset_index(drop=True)

required_validity = [
    'ndmi_valid_fraction', 'ndvi_valid_fraction', 'evi_valid_fraction',
    'nirv_valid_fraction', 'ndre_valid_fraction', 'fapar_valid_fraction',
    'par_valid_fraction', 'apar_valid_fraction'
]
missing_validity = [column for column in required_validity if column not in samples.columns]
if missing_validity:
    raise ValueError(f'Missing predictor-validity columns: {missing_validity}')

print(f'Kept {len(samples):,} chips')
display(
    samples[
        [
            'n_sif_assigned', 'n_valid_footprints', 'target_accepted_footprints',
            *required_validity, 'mean_target_modis_sif'
        ]
    ].describe().T
)

display(
    samples.groupby(
        ['sif_year', 'month', 'measurement_mode'], dropna=False
    ).size().rename('n_chips').reset_index()
)

## 5. Date-Grouped Train/Validation/Test Split

All chips from one OCO-2 acquisition date remain in one split. Dates are assigned within year, month, and measurement-mode strata when enough dates are available.

In [ ]:
def grouped_date_split(
    table: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    date_table = (
        table.groupby('Delta_Date', as_index=False)
        .agg(
            sif_year=('sif_year', 'first'),
            month=('month', 'first'),
            measurement_mode=('measurement_mode', 'first'),
            n_chips=('chip_id', 'size'),
        )
    )

    mode_count = table.groupby('Delta_Date')['measurement_mode'].nunique(dropna=False)
    if (mode_count > 1).any():
        raise ValueError('At least one Delta_Date contains multiple measurement modes.')

    rng = np.random.default_rng(SEED)
    train_dates, val_dates, test_dates = [], [], []

    for _, group in date_table.groupby(
        ['sif_year', 'month', 'measurement_mode'],
        sort=False,
        dropna=False,
    ):
        dates = group['Delta_Date'].to_numpy(copy=True)
        rng.shuffle(dates)
        n_dates = len(dates)

        if n_dates < 3:
            train_dates.extend(dates)
            continue

        n_val = max(1, int(round(n_dates * VAL_FRAC)))
        n_test = max(1, int(round(n_dates * TEST_FRAC)))

        if n_val + n_test >= n_dates:
            n_val = 1
            n_test = 1

        val_dates.extend(dates[:n_val])
        test_dates.extend(dates[n_val:n_val + n_test])
        train_dates.extend(dates[n_val + n_test:])

    train_dates = set(pd.to_datetime(train_dates))
    val_dates = set(pd.to_datetime(val_dates))
    test_dates = set(pd.to_datetime(test_dates))

    if train_dates & val_dates or train_dates & test_dates or val_dates & test_dates:
        raise RuntimeError('Date leakage detected between splits.')
    if not val_dates or not test_dates:
        raise ValueError('Grouped split produced an empty validation or test set.')

    train = table[table['Delta_Date'].isin(train_dates)].copy().reset_index(drop=True)
    val = table[table['Delta_Date'].isin(val_dates)].copy().reset_index(drop=True)
    test = table[table['Delta_Date'].isin(test_dates)].copy().reset_index(drop=True)
    return train, val, test


train_table, val_table, test_table = grouped_date_split(samples)

print('train chips:', len(train_table), 'dates:', train_table['Delta_Date'].nunique())
print('val chips:', len(val_table), 'dates:', val_table['Delta_Date'].nunique())
print('test chips:', len(test_table), 'dates:', test_table['Delta_Date'].nunique())

assert set(train_table['Delta_Date']).isdisjoint(set(val_table['Delta_Date']))
assert set(train_table['Delta_Date']).isdisjoint(set(test_table['Delta_Date']))
assert set(val_table['Delta_Date']).isdisjoint(set(test_table['Delta_Date']))

split_summary = pd.concat([
    train_table.assign(split='train'),
    val_table.assign(split='validation'),
    test_table.assign(split='test'),
]).groupby(
    ['split', 'sif_year', 'month', 'measurement_mode'],
    dropna=False,
).size().rename('n_chips').reset_index()

display(split_summary)

# Monthly Sentinel L3A products can be shared by multiple SIF dates. This is
# not date leakage, but it is reported because a stricter product/year-month
# split may be useful as a later generalization experiment.
for left_name, left_table, right_name, right_table in [
    ('train', train_table, 'validation', val_table),
    ('train', train_table, 'test', test_table),
    ('validation', val_table, 'test', test_table),
]:
    shared_products = (
        set(left_table['product_path'].astype(str))
        & set(right_table['product_path'].astype(str))
    )
    print(
        f'shared Sentinel products, {left_name} vs {right_name}: '
        f'{len(shared_products)}'
    )

## 6. Dataset, Shard Cache, and NaN-Aware Normalization

In [ ]:
class NpzShardCache:
    def __init__(self, max_size: int):
        self.max_size = max(1, int(max_size))
        self.cache: OrderedDict[str, dict[str, np.ndarray]] = OrderedDict()

    def get(self, path: str) -> dict[str, np.ndarray]:
        if path in self.cache:
            self.cache.move_to_end(path)
            return self.cache[path]

        with np.load(path, allow_pickle=False) as shard:
            loaded = {
                'X': shard['X'].copy(),
                'footprint_masks': shard['footprint_masks'].copy(),
                'y_targets': shard['y_targets'].copy(),
                'target_accept': shard['target_accept'].copy(),
                'footprint_valid': shard['footprint_valid'].copy(),
                'sif_row_ids': shard['sif_row_ids'].copy(),
            }

        self.cache[path] = loaded
        self.cache.move_to_end(path)
        while len(self.cache) > self.max_size:
            self.cache.popitem(last=False)
        return loaded


class SentinelMultiSifDataset(Dataset):
    def __init__(
        self,
        table: pd.DataFrame,
        channel_mean: np.ndarray | None = None,
        channel_std: np.ndarray | None = None,
        target_mean: float | None = None,
        target_std: float | None = None,
        cache_size: int = 8,
    ):
        self.table = table.reset_index(drop=True).copy()
        self.channel_mean = channel_mean
        self.channel_std = channel_std
        self.target_mean = target_mean
        self.target_std = target_std
        self.cache = NpzShardCache(cache_size)

    def __len__(self) -> int:
        return len(self.table)

    def __getitem__(self, index: int):
        row = self.table.iloc[index]
        shard = self.cache.get(str(row['shard_path']))
        local_index = int(row['local_index'])

        # Stored float16 arrays are cast to float32 before normalization/training.
        x = shard['X'][local_index].astype(np.float32, copy=True)
        masks = shard['footprint_masks'][local_index].astype(np.float32, copy=True)
        y = shard['y_targets'][local_index].astype(np.float32, copy=True)
        target_accept = shard['target_accept'][local_index].astype(bool)
        footprint_valid = shard['footprint_valid'][local_index].astype(bool)
        sif_row_ids = shard['sif_row_ids'][local_index].astype(np.int64, copy=True)

        mask_nonempty = masks.sum(axis=(1, 2)) > 0
        valid = footprint_valid & target_accept & np.isfinite(y) & mask_nonempty

        if self.channel_mean is not None and self.channel_std is not None:
            x = (
                x - self.channel_mean[:, None, None]
            ) / self.channel_std[:, None, None]
            # Missing raw pixels become the normalized channel mean.
            x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

        if self.target_mean is not None and self.target_std is not None:
            y[valid] = (y[valid] - self.target_mean) / self.target_std

        return (
            torch.from_numpy(x),
            torch.from_numpy(masks),
            torch.from_numpy(y),
            torch.from_numpy(valid),
            torch.from_numpy(sif_row_ids),
            torch.tensor(index, dtype=torch.long),
        )


def make_stats_cache_path(table: pd.DataFrame) -> Path:
    fields = [
        f'target={TARGET_COL}',
        f'seed={SEED}',
        f'max_stats={MAX_SAMPLES_FOR_STATS}',
        'channels=' + ','.join(channel_names),
        *sorted(table['chip_id'].astype(str).tolist()),
    ]
    digest = hashlib.md5('\n'.join(fields).encode('utf-8')).hexdigest()[:16]
    return STATS_CACHE_DIR / f'sentinel2_normalization_{digest}.npz'


def compute_channel_stats(
    table: pd.DataFrame,
    max_samples: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(SEED)
    if len(table) > max_samples:
        selected = rng.choice(len(table), size=max_samples, replace=False)
        stat_table = table.iloc[selected].copy().reset_index(drop=True)
    else:
        stat_table = table.copy().reset_index(drop=True)

    dataset = SentinelMultiSifDataset(
        stat_table,
        cache_size=SHARD_CACHE_SIZE,
    )

    n_channels = len(channel_names)
    channel_sum = np.zeros(n_channels, dtype=np.float64)
    channel_sumsq = np.zeros(n_channels, dtype=np.float64)
    channel_count = np.zeros(n_channels, dtype=np.int64)

    for index in range(len(dataset)):
        x, _, _, _, _, _ = dataset[index]
        array = x.numpy()
        finite = np.isfinite(array)
        safe = np.where(finite, array, 0.0)

        channel_sum += safe.sum(axis=(1, 2), dtype=np.float64)
        channel_sumsq += (safe * safe).sum(axis=(1, 2), dtype=np.float64)
        channel_count += finite.sum(axis=(1, 2))

        if (index + 1) % 32 == 0 or index + 1 == len(dataset):
            print(f'Normalization stats: {index + 1} / {len(dataset)} chips')

    if (channel_count == 0).any():
        bad = np.asarray(channel_names)[channel_count == 0].tolist()
        raise ValueError(f'Channels have no valid training pixels: {bad}')

    mean = channel_sum / channel_count
    variance = np.maximum(channel_sumsq / channel_count - mean ** 2, 1e-8)
    std = np.sqrt(variance)

    return mean.astype(np.float32), std.astype(np.float32), channel_count


def collect_training_targets(
    train_table: pd.DataFrame,
    footprints: pd.DataFrame,
) -> np.ndarray:
    train_chip_ids = set(train_table['chip_id'].astype(str))
    accepted = footprints[
        footprints['chip_id'].astype(str).isin(train_chip_ids)
        & (pd.to_numeric(footprints['slot'], errors='coerce') >= 0)
        & (
            footprints[FINAL_CHECK_COL].astype(str).str.strip().str.lower()
            == 'accept'
        )
    ].copy()

    values = pd.to_numeric(accepted[TARGET_COL], errors='coerce').to_numpy()
    values = values[np.isfinite(values)]
    if values.size == 0:
        raise ValueError('No accepted training targets found.')
    return values


stats_cache_path = make_stats_cache_path(train_table)

if USE_STATS_CACHE and stats_cache_path.exists():
    print('Loading normalization stats:', stats_cache_path)
    with np.load(stats_cache_path, allow_pickle=False) as stats:
        cached_channels = decode_strings(stats['channel_names'])
        if cached_channels != channel_names:
            raise ValueError('Cached channel names do not match the current dataset.')
        channel_mean = stats['channel_mean'].astype(np.float32)
        channel_std = stats['channel_std'].astype(np.float32)
        channel_count = stats['channel_count'].astype(np.int64)
        target_mean = float(stats['target_mean'])
        target_std = float(stats['target_std'])
else:
    print('Computing NaN-aware channel statistics from training chips...')
    channel_mean, channel_std, channel_count = compute_channel_stats(
        train_table,
        MAX_SAMPLES_FOR_STATS,
    )

    training_targets = collect_training_targets(train_table, footprint_metadata)
    target_mean = float(training_targets.mean())
    target_std = float(training_targets.std(ddof=1))
    if not np.isfinite(target_std) or target_std <= 0:
        raise ValueError(f'Invalid target standard deviation: {target_std}')

    np.savez_compressed(
        stats_cache_path,
        channel_names=np.asarray(channel_names),
        channel_mean=channel_mean,
        channel_std=channel_std,
        channel_count=channel_count,
        target_mean=np.asarray(target_mean, dtype=np.float32),
        target_std=np.asarray(target_std, dtype=np.float32),
    )
    print('Saved normalization stats:', stats_cache_path)

normalization_table = pd.DataFrame({
    'channel': channel_names,
    'mean': channel_mean,
    'std': channel_std,
    'valid_pixel_count': channel_count,
})
display(normalization_table)
print('target mean:', target_mean)
print('target std:', target_std)

## 7. Shard-Aware DataLoaders

The sampler shuffles shard order and sample order inside each shard while keeping batches local to a shard. This avoids repeatedly decompressing unrelated NPZ files.

In [ ]:
class ShardBatchSampler(Sampler[list[int]]):
    def __init__(
        self,
        table: pd.DataFrame,
        batch_size: int,
        shuffle: bool,
        seed: int,
    ):
        self.table = table.reset_index(drop=True)
        self.batch_size = int(batch_size)
        self.shuffle = bool(shuffle)
        self.seed = int(seed)
        self.epoch = 0
        self.groups = [
            np.asarray(indices, dtype=np.int64)
            for indices in self.table.groupby('shard_path', sort=False).indices.values()
        ]

    def set_epoch(self, epoch: int) -> None:
        self.epoch = int(epoch)

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        group_order = np.arange(len(self.groups))
        if self.shuffle:
            rng.shuffle(group_order)

        for group_index in group_order:
            indices = self.groups[group_index].copy()
            if self.shuffle:
                rng.shuffle(indices)
            for start in range(0, len(indices), self.batch_size):
                yield indices[start:start + self.batch_size].tolist()

    def __len__(self) -> int:
        return sum(
            math.ceil(len(indices) / self.batch_size)
            for indices in self.groups
        )


def seed_worker(worker_id: int) -> None:
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


dataset_kwargs = dict(
    channel_mean=channel_mean,
    channel_std=channel_std,
    target_mean=target_mean if NORMALIZE_TARGET else None,
    target_std=target_std if NORMALIZE_TARGET else None,
    cache_size=SHARD_CACHE_SIZE,
)

train_dataset = SentinelMultiSifDataset(train_table, **dataset_kwargs)
val_dataset = SentinelMultiSifDataset(val_table, **dataset_kwargs)
test_dataset = SentinelMultiSifDataset(test_table, **dataset_kwargs)

train_batch_sampler = ShardBatchSampler(
    train_table, BATCH_SIZE, shuffle=True, seed=SEED
)
val_batch_sampler = ShardBatchSampler(
    val_table, BATCH_SIZE, shuffle=False, seed=SEED
)
test_batch_sampler = ShardBatchSampler(
    test_table, BATCH_SIZE, shuffle=False, seed=SEED
)

loader_kwargs = {
    'num_workers': NUM_WORKERS,
    'pin_memory': DEVICE.type == 'cuda',
    'worker_init_fn': seed_worker,
}
if NUM_WORKERS > 0:
    loader_kwargs.update({
        'persistent_workers': True,
        'prefetch_factor': 2,
    })

train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_batch_sampler,
    **loader_kwargs,
)
val_loader = DataLoader(
    val_dataset,
    batch_sampler=val_batch_sampler,
    **loader_kwargs,
)
test_loader = DataLoader(
    test_dataset,
    batch_sampler=test_batch_sampler,
    **loader_kwargs,
)

print('train batches:', len(train_loader))
print('validation batches:', len(val_loader))
print('test batches:', len(test_loader))

## 8. Two-Level U-Net

A 300-pixel side length is divisible by four, so the two pooling stages follow `300 Ã¢â€ â€™ 150 Ã¢â€ â€™ 75` and return exactly to 300 pixels.

In [ ]:
def group_count(channels: int) -> int:
    for groups in (8, 4, 2, 1):
        if channels % groups == 0:
            return groups
    return 1


class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        groups = group_count(out_channels)
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class SmallUNet(nn.Module):
    def __init__(self, in_channels: int, base_channels: int = 16):
        super().__init__()
        self.enc1 = ConvBlock(in_channels, base_channels)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base_channels, base_channels * 2)
        self.pool2 = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(base_channels * 2, base_channels * 4)

        self.up2 = nn.ConvTranspose2d(
            base_channels * 4, base_channels * 2, kernel_size=2, stride=2
        )
        self.dec2 = ConvBlock(base_channels * 4, base_channels * 2)
        self.up1 = nn.ConvTranspose2d(
            base_channels * 2, base_channels, kernel_size=2, stride=2
        )
        self.dec1 = ConvBlock(base_channels * 2, base_channels)
        self.out = nn.Conv2d(base_channels, 1, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        bottleneck = self.bottleneck(self.pool2(e2))

        d2 = self.up2(bottleneck)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return self.out(d1)


model = SmallUNet(
    in_channels=len(channel_names),
    base_channels=BASE_CHANNELS,
).to(DEVICE)

n_parameters = sum(parameter.numel() for parameter in model.parameters())
print(model)
print(f'trainable parameters: {n_parameters:,}')

## 9. Multi-Footprint Loss, Prediction, and Metrics

In [ ]:
def masked_average_multi(
    pred_map: torch.Tensor,
    footprint_masks: torch.Tensor,
    eps: float = 1e-6,
) -> torch.Tensor:
    # pred_map: [batch, 1, height, width]
    # masks:    [batch, footprints, height, width]
    pred_float = pred_map[:, 0].float()
    mask_float = footprint_masks.float()
    numerator = (pred_float[:, None] * mask_float).sum(dim=(-2, -1))
    denominator = mask_float.sum(dim=(-2, -1)).clamp_min(eps)
    return numerator / denominator


def chip_averaged_huber(
    predictions: torch.Tensor,
    targets: torch.Tensor,
    valid: torch.Tensor,
) -> torch.Tensor:
    # Invalid padded slots contain NaN targets. Replace them before
    # evaluating Huber loss because NaN multiplied by a zero mask remains NaN.
    safe_targets = torch.where(valid, targets, predictions.detach())
    element_loss = F.smooth_l1_loss(
        predictions,
        safe_targets,
        reduction='none',
        beta=HUBER_BETA,
    )
    valid_float = valid.float()
    valid_count = valid_float.sum(dim=1)
    chip_loss = (element_loss * valid_float).sum(dim=1) / valid_count.clamp_min(1.0)
    usable_chips = valid_count > 0

    if not usable_chips.any():
        raise ValueError('Batch contains no accepted footprint targets.')
    return chip_loss[usable_chips].mean()


def denormalize_sif(values: np.ndarray) -> np.ndarray:
    if NORMALIZE_TARGET:
        return values * target_std + target_mean
    return values


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if y_true.size == 0:
        return {'n': 0, 'rmse': np.nan, 'mae': np.nan, 'bias': np.nan, 'r2': np.nan}

    residual = y_pred - y_true
    ss_res = np.sum(residual ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    r2 = np.nan if ss_tot <= 0 else 1.0 - ss_res / ss_tot

    return {
        'n': int(y_true.size),
        'rmse': float(np.sqrt(np.mean(residual ** 2))),
        'mae': float(np.mean(np.abs(residual))),
        'bias': float(np.mean(residual)),
        'r2': float(r2),
    }


footprint_lookup_columns = [
    'chip_id', 'sif_row_id', 'state', 'hzs', 'measurement_mode'
]
available_lookup_columns = [
    column for column in footprint_lookup_columns
    if column in footprint_metadata.columns
]
footprint_lookup = (
    footprint_metadata[available_lookup_columns]
    .drop_duplicates(['chip_id', 'sif_row_id'])
)


@torch.no_grad()
def predict_table(
    model: nn.Module,
    loader: DataLoader,
    dataset: SentinelMultiSifDataset,
) -> pd.DataFrame:
    model.eval()
    rows = []

    for x, masks, y, valid, sif_row_ids, sample_indices in loader:
        x = x.to(DEVICE, non_blocking=True)
        masks_gpu = masks.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=AMP_ENABLED,
        ):
            pred_map = model(x)

        pred_normalized = masked_average_multi(pred_map, masks_gpu).cpu().numpy()
        y_normalized = y.numpy()
        valid_array = valid.numpy().astype(bool)
        sif_ids = sif_row_ids.numpy()
        sample_indices = sample_indices.numpy()

        pred_sif = denormalize_sif(pred_normalized)
        observed_sif = denormalize_sif(y_normalized)

        for batch_index, sample_index in enumerate(sample_indices):
            sample = dataset.table.iloc[int(sample_index)]
            for slot in np.flatnonzero(valid_array[batch_index]):
                rows.append({
                    'chip_id': sample['chip_id'],
                    'slot': int(slot),
                    'sif_row_id': int(sif_ids[batch_index, slot]),
                    'Delta_Date': sample['Delta_Date'],
                    'sif_year': int(sample['sif_year']),
                    'sif_doy': int(sample['sif_doy']),
                    'month': int(sample['month']),
                    'mgrs_tile_t': sample['mgrs_tile_t'],
                    'chip_measurement_mode': sample['measurement_mode'],
                    'observed_sif': float(observed_sif[batch_index, slot]),
                    'predicted_sif_raw': float(pred_sif[batch_index, slot]),
                })

    predictions = pd.DataFrame(rows)
    predictions = predictions.merge(
        footprint_lookup,
        on=['chip_id', 'sif_row_id'],
        how='left',
        validate='many_to_one',
    )
    return predictions


def fit_linear_calibration(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    design = np.column_stack([np.ones(valid.sum()), y_pred[valid]])
    intercept, slope = np.linalg.lstsq(design, y_true[valid], rcond=None)[0]
    return {'intercept': float(intercept), 'slope': float(slope)}


def apply_linear_calibration(
    y_pred: np.ndarray,
    calibration: dict[str, float],
) -> np.ndarray:
    return calibration['intercept'] + calibration['slope'] * np.asarray(y_pred)

## 10. Training

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=LR_FACTOR,
    patience=LR_PATIENCE,
    min_lr=MIN_LEARNING_RATE,
)
scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

history = []
best_val_rmse = np.inf
best_epoch = 0
best_state = None
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_batch_sampler.set_epoch(epoch)
    model.train()
    optimizer.zero_grad(set_to_none=True)

    epoch_losses = []

    for step, (x, masks, y, valid, _, _) in enumerate(train_loader, start=1):
        x = x.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        valid = valid.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=AMP_ENABLED,
        ):
            pred_map = model(x)
            pred_sif = masked_average_multi(pred_map, masks)
            loss = chip_averaged_huber(pred_sif, y, valid)
            accumulated_loss = loss / GRADIENT_ACCUMULATION_STEPS

        scaler.scale(accumulated_loss).backward()

        should_step = (
            step % GRADIENT_ACCUMULATION_STEPS == 0
            or step == len(train_loader)
        )
        if should_step:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        epoch_losses.append(float(loss.detach().cpu()))

    val_predictions_epoch = predict_table(model, val_loader, val_dataset)
    val_metrics = regression_metrics(
        val_predictions_epoch['observed_sif'],
        val_predictions_epoch['predicted_sif_raw'],
    )
    train_loss = float(np.mean(epoch_losses))
    current_lr = float(optimizer.param_groups[0]['lr'])
    scheduler.step(val_metrics['rmse'])

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_rmse': val_metrics['rmse'],
        'val_mae': val_metrics['mae'],
        'val_bias': val_metrics['bias'],
        'val_r2': val_metrics['r2'],
        'learning_rate': current_lr,
    })

    print(
        f"Epoch {epoch:03d} | train_huber={train_loss:.5f} | "
        f"val_rmse={val_metrics['rmse']:.5f} | "
        f"val_r2={val_metrics['r2']:.4f} | "
        f"val_mae={val_metrics['mae']:.5f} | "
        f"val_bias={val_metrics['bias']:.5f} | "
        f"lr={current_lr:.2e}"
    )

    if val_metrics['rmse'] < best_val_rmse:
        best_val_rmse = val_metrics['rmse']
        best_epoch = epoch
        best_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor in model.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f'Early stopping at epoch {epoch}; best epoch was {best_epoch}.')
        break

if best_state is None:
    raise RuntimeError('No best model state was recorded.')

model.load_state_dict(best_state)
history_df = pd.DataFrame(history)
print('best epoch:', best_epoch)
print('best validation RMSE:', best_val_rmse)

## 11. Validation Calibration and Final Test Evaluation

In [ ]:
val_predictions = predict_table(model, val_loader, val_dataset)
test_predictions = predict_table(model, test_loader, test_dataset)

val_metrics_raw = regression_metrics(
    val_predictions['observed_sif'],
    val_predictions['predicted_sif_raw'],
)
test_metrics_raw = regression_metrics(
    test_predictions['observed_sif'],
    test_predictions['predicted_sif_raw'],
)

calibration = fit_linear_calibration(
    val_predictions['observed_sif'].to_numpy(),
    val_predictions['predicted_sif_raw'].to_numpy(),
)

for table in (val_predictions, test_predictions):
    table['predicted_sif_calibrated'] = apply_linear_calibration(
        table['predicted_sif_raw'].to_numpy(),
        calibration,
    )
    table['predicted_sif'] = table['predicted_sif_calibrated']
    table['residual_raw'] = (
        table['predicted_sif_raw'] - table['observed_sif']
    )
    table['residual_calibrated'] = (
        table['predicted_sif_calibrated'] - table['observed_sif']
    )
    table['residual'] = table['residual_calibrated']

val_metrics_calibrated = regression_metrics(
    val_predictions['observed_sif'],
    val_predictions['predicted_sif_calibrated'],
)
test_metrics_calibrated = regression_metrics(
    test_predictions['observed_sif'],
    test_predictions['predicted_sif_calibrated'],
)

print('Validation metrics raw:')
print(val_metrics_raw)
print('\nValidation metrics calibrated:')
print(val_metrics_calibrated)
print('\nTest metrics raw:')
print(test_metrics_raw)
print('\nTest metrics calibrated:')
print(test_metrics_calibrated)
print('\nValidation calibration:')
print(calibration)

display(test_predictions.head())

## 12. Metrics by SIF Bin, Measurement Mode, Month, MGRS Tile, and Hardiness Zone

In [ ]:
test_predictions['sif_bin'] = pd.cut(
    test_predictions['observed_sif'],
    bins=SIF_BIN_EDGES,
    labels=SIF_BIN_LABELS,
    right=False,
    include_lowest=True,
)


def metrics_by_group(
    table: pd.DataFrame,
    group_column: str,
) -> pd.DataFrame:
    rows = []
    for group_value, group in table.groupby(group_column, dropna=False, observed=False):
        raw = regression_metrics(
            group['observed_sif'],
            group['predicted_sif_raw'],
        )
        calibrated = regression_metrics(
            group['observed_sif'],
            group['predicted_sif_calibrated'],
        )
        rows.append({
            'group_variable': group_column,
            'group_value': str(group_value),
            'n': raw['n'],
            'observed_min': float(group['observed_sif'].min()),
            'observed_max': float(group['observed_sif'].max()),
            'observed_mean': float(group['observed_sif'].mean()),
            'raw_predicted_mean': float(group['predicted_sif_raw'].mean()),
            'raw_rmse': raw['rmse'],
            'raw_mae': raw['mae'],
            'raw_bias': raw['bias'],
            'raw_r2': raw['r2'],
            'calibrated_predicted_mean': float(
                group['predicted_sif_calibrated'].mean()
            ),
            'calibrated_rmse': calibrated['rmse'],
            'calibrated_mae': calibrated['mae'],
            'calibrated_bias': calibrated['bias'],
            'calibrated_r2': calibrated['r2'],
        })
    return pd.DataFrame(rows)


group_columns = [
    'sif_bin',
    'measurement_mode',
    'month',
    'mgrs_tile_t',
    'hzs',
]

group_metric_tables = []
for column in group_columns:
    if column in test_predictions.columns:
        group_metric_tables.append(metrics_by_group(test_predictions, column))

group_metrics = pd.concat(group_metric_tables, ignore_index=True)
display(group_metrics[group_metrics['group_variable'] == 'sif_bin'])
display(group_metrics[group_metrics['group_variable'] == 'measurement_mode'])
display(group_metrics[group_metrics['group_variable'] == 'month'])
display(group_metrics[group_metrics['group_variable'] == 'mgrs_tile_t'])
display(group_metrics[group_metrics['group_variable'] == 'hzs'])

## 13. Training and Prediction Diagnostics

In [ ]:
history_long = history_df.melt(
    id_vars=['epoch'],
    value_vars=['train_loss', 'val_rmse'],
    var_name='metric',
    value_name='value',
)

history_chart = (
    alt.Chart(history_long)
    .mark_line(point=True)
    .encode(
        x=alt.X('epoch:Q', title='Epoch'),
        y=alt.Y('value:Q', title='Metric value'),
        color=alt.Color('metric:N', title=None),
    )
    .properties(width=700, height=330, title='Training history')
)
display(history_chart)

observed = test_predictions['observed_sif'].to_numpy()
calibrated_predicted = test_predictions['predicted_sif_calibrated'].to_numpy()
axis_min = math.floor(
    min(np.nanmin(observed), np.nanmin(calibrated_predicted)) * 4
) / 4
axis_max = math.ceil(
    max(np.nanmax(observed), np.nanmax(calibrated_predicted)) * 4
) / 4

density = (
    alt.Chart(test_predictions)
    .mark_rect()
    .encode(
        x=alt.X(
            'observed_sif:Q',
            bin=alt.Bin(maxbins=60),
            scale=alt.Scale(domain=[axis_min, axis_max]),
            title='Observed OCO-2 SIF',
        ),
        y=alt.Y(
            'predicted_sif_calibrated:Q',
            bin=alt.Bin(maxbins=60),
            scale=alt.Scale(domain=[axis_min, axis_max]),
            title='Calibrated footprint-mean SIF',
        ),
        color=alt.Color(
            'count():Q',
            scale=alt.Scale(type='log', scheme='turbo'),
            title='Density',
        ),
    )
)

diagonal_data = pd.DataFrame({
    'x': [axis_min, axis_max],
    'y': [axis_min, axis_max],
})
diagonal = (
    alt.Chart(diagonal_data)
    .mark_line(color='black', strokeWidth=1.5)
    .encode(x='x:Q', y='y:Q')
)

density_title = (
    f"Test RMSE={test_metrics_calibrated['rmse']:.4f}, "
    f"RÃ‚Â²={test_metrics_calibrated['r2']:.3f}"
)
display((density + diagonal).properties(width=620, height=620, title=density_title))

residual_chart = (
    alt.Chart(test_predictions)
    .mark_bar()
    .encode(
        x=alt.X(
            'residual_calibrated:Q',
            bin=alt.Bin(maxbins=80),
            title='Calibrated predicted - observed SIF',
        ),
        y=alt.Y('count():Q', title='Count'),
    )
    .properties(width=700, height=320, title='Test residual distribution')
)
display(residual_chart)

## 14. Example Predicted SIF Map and Footprint Masks

In [ ]:
example_index = 0
raw_example_dataset = SentinelMultiSifDataset(
    test_table.iloc[[example_index]].reset_index(drop=True),
    cache_size=1,
)
normalized_example_dataset = SentinelMultiSifDataset(
    test_table.iloc[[example_index]].reset_index(drop=True),
    **dataset_kwargs,
)

raw_x, raw_masks, raw_y, raw_valid, raw_ids, _ = raw_example_dataset[0]
norm_x, norm_masks, norm_y, norm_valid, norm_ids, _ = normalized_example_dataset[0]

model.eval()
with torch.no_grad():
    with torch.autocast(
        device_type=DEVICE.type,
        dtype=torch.float16,
        enabled=AMP_ENABLED,
    ):
        example_pred_norm = model(norm_x[None].to(DEVICE))[0, 0]
    example_pred_map_raw = denormalize_sif(
        example_pred_norm.float().cpu().numpy()
    )
    # Linear calibration is affine, so applying it pixel-wise is consistent
    # with applying it after footprint averaging.
    example_pred_map = apply_linear_calibration(
        example_pred_map_raw,
        calibration,
    )

ndvi_index = channel_names.index('ndvi')
combined_mask = raw_masks[norm_valid].sum(dim=0).numpy()

figure, axes = plt.subplots(1, 3, figsize=(18, 5.5), constrained_layout=True)

image0 = axes[0].imshow(example_pred_map, cmap='viridis')
axes[0].set_title('Calibrated predicted 20 m SIF map')
figure.colorbar(image0, ax=axes[0], fraction=0.046)

image1 = axes[1].imshow(raw_x[ndvi_index], cmap='RdYlGn', vmin=-1, vmax=1)
axes[1].set_title('Raw NDVI channel')
figure.colorbar(image1, ax=axes[1], fraction=0.046)

image2 = axes[2].imshow(combined_mask, cmap='magma')
axes[2].set_title(f'Combined footprint masks (n={int(norm_valid.sum())})')
figure.colorbar(image2, ax=axes[2], fraction=0.046)

for axis in axes:
    axis.set_xticks([])
    axis.set_yticks([])

plt.show()

example_footprint_predictions = masked_average_multi(
    torch.from_numpy(example_pred_map)[None, None].to(DEVICE),
    norm_masks[None].to(DEVICE),
)[0].cpu().numpy()

example_rows = []
for slot in np.flatnonzero(norm_valid.numpy()):
    example_rows.append({
        'slot': int(slot),
        'sif_row_id': int(norm_ids[slot]),
        'observed_sif': float(denormalize_sif(norm_y[slot].numpy())),
        'predicted_sif_calibrated': float(example_footprint_predictions[slot]),
    })
display(pd.DataFrame(example_rows))

## 15. Save Model, Predictions, Splits, and Metrics

In [ ]:
history_path = OUTPUT_DIR / 'training_history.csv'
val_predictions_path = OUTPUT_DIR / 'validation_predictions.csv'
test_predictions_path = OUTPUT_DIR / 'test_predictions.csv'
group_metrics_path = OUTPUT_DIR / 'test_group_metrics.csv'
split_path = OUTPUT_DIR / 'chip_splits.csv'
checkpoint_path = OUTPUT_DIR / 'sentinel2_multisif_unet.pt'
metrics_path = OUTPUT_DIR / 'metrics.json'

history_df.to_csv(history_path, index=False)
val_predictions.to_csv(val_predictions_path, index=False)
test_predictions.to_csv(test_predictions_path, index=False)
group_metrics.to_csv(group_metrics_path, index=False)

chip_splits = pd.concat([
    train_table[['chip_id', 'Delta_Date']].assign(split='train'),
    val_table[['chip_id', 'Delta_Date']].assign(split='validation'),
    test_table[['chip_id', 'Delta_Date']].assign(split='test'),
], ignore_index=True)
chip_splits.to_csv(split_path, index=False)

checkpoint = {
    'model_state_dict': best_state,
    'model_class': 'SmallUNet',
    'channel_names': channel_names,
    'target_column': TARGET_COL,
    'final_check_column': FINAL_CHECK_COL,
    'base_channels': BASE_CHANNELS,
    'channel_mean': channel_mean,
    'channel_std': channel_std,
    'target_mean': target_mean,
    'target_std': target_std,
    'normalize_target': NORMALIZE_TARGET,
    'calibration': calibration,
    'best_epoch': best_epoch,
    'best_validation_rmse': best_val_rmse,
    'config': {
        'batch_size': BATCH_SIZE,
        'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
        'epochs': EPOCHS,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'huber_beta': HUBER_BETA,
        'seed': SEED,
        'date_grouped_split': True,
        'max_samples_for_stats': MAX_SAMPLES_FOR_STATS,
        'amp_enabled': AMP_ENABLED,
    },
}
torch.save(checkpoint, checkpoint_path)

metrics_payload = {
    'best_epoch': best_epoch,
    'best_validation_rmse_during_training': best_val_rmse,
    'validation_raw': val_metrics_raw,
    'validation_calibrated': val_metrics_calibrated,
    'test_raw': test_metrics_raw,
    'test_calibrated': test_metrics_calibrated,
    'calibration': calibration,
    'n_train_chips': len(train_table),
    'n_validation_chips': len(val_table),
    'n_test_chips': len(test_table),
    'n_train_dates': int(train_table['Delta_Date'].nunique()),
    'n_validation_dates': int(val_table['Delta_Date'].nunique()),
    'n_test_dates': int(test_table['Delta_Date'].nunique()),
    'stats_cache_path': str(stats_cache_path),
}
with metrics_path.open('w', encoding='utf-8') as file:
    json.dump(metrics_payload, file, indent=2)

print('Saved:')
for path in [
    history_path,
    val_predictions_path,
    test_predictions_path,
    group_metrics_path,
    split_path,
    checkpoint_path,
    metrics_path,
]:
    print(' -', path)